In [1]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

import pandas as pd

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

df = pd.read_csv(url,names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
cols_with_missing_vals = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0,np.nan)

df.fillna(df.mean(),inplace=True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [3]:
x = df.drop('Outcome',axis = 1)
y = df['Outcome']

X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state = 42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)



In [4]:
X_train.shape

(537, 8)

In [5]:
X_test.shape

(231, 8)

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_Depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_Depth,
        random_state=42
    )

    score = cross_val_score(model,X_train,y_train,cv =3,scoring='accuracy').mean()
    return score

In [7]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2026-07-13 19:04:02,831] A new study created in memory with name: no-name-e897ffdf-4b62-43ed-af65-a51e8baf6bdc
[I 2026-07-13 19:04:03,168] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 123, 'max_depth': 14}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-13 19:04:03,381] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 76, 'max_depth': 15}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-13 19:04:03,852] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 154, 'max_depth': 13}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-13 19:04:04,107] Trial 3 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 105, 'max_depth': 3}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-07-13 19:04:04,574] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 177, 'max_depth': 9}. Best is trial 0 with value: 0.7728119

In [8]:
study.best_trial.value

0.7802607076350093

In [9]:
study.best_trial.params

{'n_estimators': 117, 'max_depth': 16}

In [10]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params,random_state=42)

best_model.fit(X_train,y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test,y_pred)
print(f"Test accuracy with beest hyperparameters:{test_accuracy:.2f}")



Test accuracy with beest hyperparameters:0.73


### Samplers

random sampler

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective1(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_Depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_Depth,
        random_state=42
    )

    score = cross_val_score(model,X_train,y_train,cv =3,scoring='accuracy').mean()
    return score

In [12]:
study1 = optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler())
study1.optimize(objective1,n_trials=50)

[I 2026-07-13 19:04:23,554] A new study created in memory with name: no-name-37d8d0f2-b90d-4677-b31f-585dec31d1c3
[I 2026-07-13 19:04:23,897] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 123, 'max_depth': 11}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-13 19:04:24,150] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 91, 'max_depth': 13}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-13 19:04:24,608] Trial 2 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 168, 'max_depth': 16}. Best is trial 2 with value: 0.7765363128491621.
[I 2026-07-13 19:04:24,998] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 143, 'max_depth': 13}. Best is trial 2 with value: 0.7765363128491621.
[I 2026-07-13 19:04:25,198] Trial 4 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 71, 'max_depth': 20}. Best is trial 4 with value: 0.778398

In [13]:
study1.best_trial.value

0.7839851024208566

In [14]:
study1.best_trial.params

{'n_estimators': 73, 'max_depth': 18}

In [15]:
best_model1 = RandomForestClassifier(**study1.best_trial.params,random_state=42)

best_model1.fit(X_train,y_train)

y_pred1 = best_model1.predict(X_test)

test_accuracy1 = accuracy_score(y_test,y_pred1)

print(f'Test accuracy on best hyperparameters:{test_accuracy1:.2f}')

Test accuracy on best hyperparameters:0.74


grid search

In [16]:
search_space = {
    'n_estimators' : [50,100,150,200],
    'max_depth' : [5,10,15,20]
}

In [17]:
study2 = optuna.create_study(direction='maximize',sampler=optuna.samplers.GridSampler(search_space))
study2.optimize(objective1)

[I 2026-07-13 19:04:40,974] A new study created in memory with name: no-name-173e2238-13c2-4a6f-9704-659438ffd6f3
[I 2026-07-13 19:04:41,227] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-07-13 19:04:41,632] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-07-13 19:04:41,775] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-13 19:04:42,050] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-13 19:04:42,327] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [18]:
study2.best_trial.value

0.7746741154562384

In [19]:
study2.best_trial.params

{'n_estimators': 50, 'max_depth': 5}

In [20]:
best_model2 = RandomForestClassifier(**study1.best_trial.params,random_state=42)

best_model2.fit(X_train,y_train)

y_pred2 = best_model2.predict(X_test)

test_accuracy2 = accuracy_score(y_test,y_pred2)

print(f'Test accuracy with best hyperparameters:{test_accuracy2:.2f}')

Test accuracy with best hyperparameters:0.74


### Optuna visualization


In [21]:
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_contour,plot_param_importances

In [22]:
import plotly

##### optimization_history

In [23]:
plot_optimization_history(study).show()

##### Parallel coordinate plot

In [24]:
plot_parallel_coordinate(study).show()

##### Slice plot

In [25]:
plot_slice(study).show()

##### Contour plot

In [26]:
plot_contour(study).show()

##### Importance plot

In [27]:
plot_param_importances(study).show()

### Define by run feature

In [28]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

In [34]:
def objective2(trial):

    classifier_name = trial.suggest_categorical('classifier',['SVM','RandomForest','GradientBoosting'])

    if classifier_name == 'SVM':
        c = trial.suggest_float('C',0.1,100,log =True)
        kernel = trial.suggest_categorical('kernel',['linear','rbf','poly','sigmoid'])
        gamma = trial.suggest_categorical('gamma',['scale','auto'])

        model = SVC(C=c,kernel=kernel,gamma=gamma,random_state=42)

    elif classifier_name == 'RandomForest':
        n_estimators = trial.suggest_int('n_estimators',50,300)
        max_depth = trial.suggest_int('max_depth',3,20)
        min_samples_split = trial.suggest_int('min_samples_split',2,10)
        min_samples_leaf = trial.suggest_int('min_sapmles_leaf',1,10)
        bootstrap = trial.suggest_categorical('bootstrap',[True,False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        n_estimators = trial.suggest_int('n_esttimators',50,300)
        learning_rate = trial.suggest_float('learning_rate',0.01,0.3,log =True)
        max_depth = trial.suggest_int('max_depth',3,20)
        min_samples_split = trial.suggest_int('min_samples_split',2,10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',1,10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf
        )

    score = cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()
    return score

In [35]:
study3 = optuna.create_study(direction='maximize')
study3.optimize(objective2,n_trials=100)

[I 2026-07-13 19:07:57,416] A new study created in memory with name: no-name-f89ec3a1-0164-4d0d-85d4-396df7e86462
[I 2026-07-13 19:07:57,430] Trial 0 finished with value: 0.6927374301675977 and parameters: {'classifier': 'SVM', 'C': 66.97645261714807, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.6927374301675977.
[I 2026-07-13 19:07:57,450] Trial 1 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 2.918701307709879, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 1 with value: 0.7858472998137801.
[I 2026-07-13 19:07:58,154] Trial 2 finished with value: 0.7541899441340782 and parameters: {'classifier': 'RandomForest', 'n_estimators': 284, 'max_depth': 7, 'min_samples_split': 9, 'min_sapmles_leaf': 9, 'bootstrap': True}. Best is trial 1 with value: 0.7858472998137801.
[I 2026-07-13 19:07:59,271] Trial 3 finished with value: 0.7597765363128492 and parameters: {'classifier': 'GradientBoosting', 'n_esttimators': 219, 'learning_ra

In [36]:
study3.best_trial.value

0.7895716945996275

In [37]:
study3.best_trial.params

{'classifier': 'SVM',
 'C': 0.1451857289896656,
 'kernel': 'linear',
 'gamma': 'scale'}

In [39]:
study3.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_min_sapmles_leaf,params_n_estimators,params_n_esttimators,state
0,0,0.692737,2026-07-13 19:07:57.416424,2026-07-13 19:07:57.430609,0 days 00:00:00.014185,66.976453,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.785847,2026-07-13 19:07:57.430609,2026-07-13 19:07:57.450842,0 days 00:00:00.020233,2.918701,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.754190,2026-07-13 19:07:57.451847,2026-07-13 19:07:58.154017,0 days 00:00:00.702170,NaN,True,RandomForest,NaN,NaN,NaN,7.0,NaN,9.0,9.0,284.0,NaN,COMPLETE
3,3,0.759777,2026-07-13 19:07:58.154017,2026-07-13 19:07:59.271462,0 days 00:00:01.117445,NaN,NaN,GradientBoosting,NaN,NaN,0.223221,7.0,2.0,7.0,NaN,NaN,219.0,COMPLETE
4,4,0.737430,2026-07-13 19:07:59.271462,2026-07-13 19:07:59.683324,0 days 00:00:00.411862,NaN,NaN,GradientBoosting,NaN,NaN,0.192611,20.0,8.0,9.0,NaN,NaN,67.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2026-07-13 19:08:14.661394,2026-07-13 19:08:14.677275,0 days 00:00:00.015881,0.249399,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2026-07-13 19:08:14.678283,2026-07-13 19:08:14.761584,0 days 00:00:00.083301,35.075957,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.785847,2026-07-13 19:08:14.762582,2026-07-13 19:08:14.776607,0 days 00:00:00.014025,0.192865,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.731844,2026-07-13 19:08:14.777598,2026-07-13 19:08:15.694994,0 days 00:00:00.917396,NaN,NaN,GradientBoosting,NaN,NaN,0.098546,20.0,8.0,6.0,NaN,NaN,148.0,COMPLETE


to see which algo ran for how many times

In [40]:
study3.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
GradientBoosting    11
RandomForest        10
Name: count, dtype: int64

to see avg accuracy of each algo

In [41]:
study3.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.740139
RandomForest        0.761825
SVM                 0.776866
Name: value, dtype: float64

In [42]:
plot_optimization_history(study3).show()